# Capstone Showdown — Fashion-MNIST
**Objective:** Compare five model types using only our `my_ml_lib` implementation and report the best configuration for each.  
Models:
1. OvR Logistic Regression (binary logistic in One-vs-Rest)
2. Softmax Regression (Raw pixels) — autograd
3. Softmax Regression + Polynomial Features — autograd
4. Softmax Regression + Gaussian Basis Features — autograd
5. MLP (Sequential with ReLU) — autograd

In [ ]:
# Basic imports
import os
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

# Import modules from your library (only my_ml_lib)
from my_ml_lib.datasets import load_fashion_mnist
from my_ml_lib.model_selection import train_test_val_split
from my_ml_lib.preprocessing import StandardScaler, PolynomialFeatures, GaussianBasisFeatures
from my_ml_lib.linear_models.classification import LogisticRegression

# Autograd / NN
from my_ml_lib.nn.modules import Linear, ReLU, Sigmoid, Softmax, Sequential
from my_ml_lib.nn.losses import CrossEntropyLoss, BinaryCrossEntropyLoss
from my_ml_lib.nn.optim import SGD
from my_ml_lib.nn.autograd import Value

# Utility
import pickle


In [ ]:
# Paths and seeds
DATA_FOLDER = "data"
TRAIN_FILENAME = "fashion-mnist_train.csv"
TEST_FILENAME = "fashion-mnist_test.csv"
SEED = 42
np.random.seed(SEED)

# Splits
TRAIN_PROP = 0.8
VAL_PROP = 0.2

# Training hyperparameters
EPOCHS = 20
BATCH_SIZE = 128
LR_AUTOGRAD = 0.05
LR_LOGISTIC = 0.01

ALPHAS = [0.01, 0.1, 1, 10]
# Feature transforms candidates to try
POLY_DEGREES = [1]
GAUSSIAN_CENTERS = [50, 100]
GAUSSIAN_SIGMAS = [5.0, 10.0]

# Save best models
SAVED_MODELS_DIR = "saved_models"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)


### Load Fashion-MNIST (60k training CSV) and split into train / validation
- We'll load training CSV (60k rows).
- Normalize pixels to [0,1] while loading (loader supports normalize).
- Use `train_test_val_split` to get train (~48k) and val (~12k).
- Also load the provided test CSV (10k).


In [ ]:
# Load 60k train data
X_all, y_all = load_fashion_mnist(data_folder=DATA_FOLDER, train_filename=TRAIN_FILENAME, test_filename=TEST_FILENAME, kind='train', normalize=True)
print("Loaded training file shape:", X_all.shape, y_all.shape)

# Split into train and validation
X_train, X_val, X_unused, y_train, y_val, y_unused = train_test_val_split(X_all, y_all, train_size=TRAIN_PROP, val_size=VAL_PROP, test_size=1-TRAIN_PROP-VAL_PROP, shuffle=True, random_state=SEED)

print("Train shape:", X_train.shape, y_train.shape)
print("Val   shape:", X_val.shape, y_val.shape)

X_test_full, y_test_full = load_fashion_mnist(data_folder=DATA_FOLDER, train_filename=TRAIN_FILENAME, test_filename=TEST_FILENAME, kind='test', normalize=True)
print("Provided test set (10k):", X_test_full.shape, y_test_full.shape)


In [ ]:
# Utility: batching generator for autograd models (X as numpy arrays)
def create_batches(X, y, batch_size, shuffle=True, rng=None):
    n = X.shape[0]
    indices = np.arange(n)
    if shuffle:
        if rng is None:
            rng = np.random.RandomState()
        rng.shuffle(indices)
    for start in range(0, n, batch_size):
        batch_idx = indices[start:start+batch_size]
        yield X[batch_idx], y[batch_idx]

# Utility: train loop for autograd models (model: Module)
def train_autograd(model, X_train, y_train, X_val, y_val, loss_fn, lr=0.01, epochs=10, batch_size=128, verbose=False):
    optimizer = SGD(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    rng = np.random.RandomState(SEED)

    for epoch in range(epochs):
        model.zero_grad()
        train_losses = []
        for Xb, yb in create_batches(X_train, y_train, batch_size, shuffle=True, rng=rng):
            xb_val = Value(Xb)
            logits = model(xb_val)
            loss = loss_fn(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.data) if np.isscalar(loss.data) else float(np.array(loss.data).mean()))

        val_losses = []
        val_preds = []
        for Xb, yb in create_batches(X_val, y_val, batch_size, shuffle=False):
            xb_val = Value(Xb)
            logits = model(xb_val)
            loss = loss_fn(logits, yb)

            probs = logits.data
            preds = np.argmax(probs, axis=1)
            val_preds.append(preds)
            val_losses.append(float(loss.data) if np.isscalar(loss.data) else float(np.array(loss.data).mean()))
        val_preds = np.concatenate(val_preds, axis=0)
        val_acc = np.mean(val_preds == y_val[:len(val_preds)])
        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(np.mean(val_losses))
        history['val_acc'].append(val_acc)
        if verbose:
            print(f"Epoch {epoch+1}/{epochs}  train_loss={history['train_loss'][-1]:.4f}  val_loss={history['val_loss'][-1]:.4f}  val_acc={val_acc:.4f}")
    return history

# Utility: evaluate autograd model on dataset and return accuracy
def eval_autograd(model, X, y, batch_size=128):
    preds = []
    for Xb, _ in create_batches(X, y, batch_size, shuffle=False):
        xb_val = Value(Xb)
        logits = model(xb_val)
        preds.append(np.argmax(logits.data, axis=1))
    preds = np.concatenate(preds, axis=0)
    return float(np.mean(preds == y[:len(preds)]))


### Model 1 — One-vs-Rest (OvR) Logistic Regression
We will train 10 binary logistic classifiers (one per class). For each alpha in ALPHAS we:
- Train 10 classifiers on training set
- Evaluate average validation accuracy
Select best alpha and retrain the final OvR on the training set (or keep the trained one).
We save the set of classifiers via `pickle`.


In [ ]:
from copy import deepcopy

def train_ovr_logistic(X_train, y_train, X_val, y_val, alphas, max_iter=100, lr=LR_LOGISTIC):
    best_alpha = None
    best_val = -np.inf
    results = {}
    for alpha in alphas:
        val_accs = []
        # Train 10 classifiers (OvR)
        for k in range(10):
            ybin = (y_train == k).astype(int)
            clf = LogisticRegression(alpha=alpha, lr=lr, fit_intercept=True)
            clf.fit(X_train, ybin)
            # validation
            yval_bin = (y_val == k).astype(int)
            preds = clf.predict(X_val)
            val_accs.append(np.mean(preds == yval_bin))
        mean_val = np.mean(val_accs)
        results[alpha] = mean_val
        print(f"alpha={alpha}  mean OvR validation acc (avg across binary classifiers)={mean_val:.4f}")
        if mean_val > best_val:
            best_val = mean_val
            best_alpha = alpha
    # After choosing best_alpha, train final OvR classifiers on TRAIN set
    ovr_classifiers = []
    for k in range(10):
        ybin = (y_train == k).astype(int)
        clf = LogisticRegression(alpha=best_alpha, lr=lr, fit_intercept=True)
        clf.fit(X_train, ybin)
        ovr_classifiers.append(clf)
    return best_alpha, results, ovr_classifiers

best_alpha_ovr, ovr_results, ovr_classifiers = train_ovr_logistic(X_train, y_train, X_val, y_val, ALPHAS, max_iter=200)
print("Chosen best alpha (OvR):", best_alpha_ovr)


In [ ]:
# Helper to predict with OvR classifiers (returns class with max probability)
def ovr_predict(classifiers, X):
    probs = np.stack([clf.predict_proba(X)[:,1] for clf in classifiers], axis=1)  # shape (n,10)
    return np.argmax(probs, axis=1)

# Validation accuracy
ovr_val_preds = ovr_predict(ovr_classifiers, X_val)
ovr_val_acc = np.mean(ovr_val_preds == y_val)
print("OvR validation accuracy:", ovr_val_acc)

# Evaluate on the 10k test provided (optional)
ovr_test_preds = ovr_predict(ovr_classifiers, X_test_full)
ovr_test_acc = np.mean(ovr_test_preds == y_test_full)
print("OvR accuracy on provided 10k test:", ovr_test_acc)

# Save OvR models (pickle)
ovr_pickle_path = os.path.join(SAVED_MODELS_DIR, "ovr_logistic.pkl")
with open(ovr_pickle_path, "wb") as f:
    pickle.dump(ovr_classifiers, f)
print("OvR classifiers saved to", ovr_pickle_path)


### Model 2 — Softmax Regression (Raw pixels)
A single linear layer trained with CrossEntropyLoss using our autograd engine.
We tune learning rates and epochs and pick the best on validation set.


In [ ]:
n_features = X_train.shape[1]
n_classes = 10

def build_softmax_model(input_dim):
    return Sequential(Linear(input_dim, n_classes))

lr_grid = [0.05, 0.1]

softmax_results = {}
best_softmax = None
best_softmax_history = None
best_softmax_acc = -np.inf
best_softmax_config = None

for lr in lr_grid:
    model = build_softmax_model(n_features)
    loss_fn = CrossEntropyLoss(reduction='mean')
    history = train_autograd(model, X_train, y_train, X_val, y_val,
                             loss_fn, lr=lr, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True)
    val_acc = history['val_acc'][-1]
    softmax_results[lr] = val_acc
    print(f"lr={lr} val_acc={val_acc:.4f}")
    if val_acc > best_softmax_acc:
        best_softmax_acc = val_acc
        best_softmax = model
        best_softmax_history = history
        best_softmax_config = {'lr': lr}

print("Best softmax config:", best_softmax_config, "val_acc:", best_softmax_acc)
# Save best softmax model
if best_softmax is not None:
    best_softmax.save_state_dict(os.path.join(SAVED_MODELS_DIR, "softmax_raw.npz"))
    print("Saved softmax raw model state.")


### Model 3 — Softmax Regression + Polynomial Features
We apply polynomial expansion (try a small set of degrees) and train the same softmax classifier on transformed features.
Polynomial expansion may create many features; keep degree small.


In [ ]:
poly_results = {}
best_poly_model = None
best_poly_history = None
best_poly_acc = -np.inf
best_poly_config = None

for degree in POLY_DEGREES:
    print(f"\n--- Degree {degree} ---")
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_val_poly = poly.transform(X_val)
    print("Transformed shapes:", X_train_poly.shape, X_val_poly.shape)
    # train
    model = build_softmax_model(X_train_poly.shape[1])
    loss_fn = CrossEntropyLoss(reduction='mean')
    history = train_autograd(model, X_train_poly, y_train, X_val_poly, y_val,
                             loss_fn, lr=LR_AUTOGRAD, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True)
    val_acc = history['val_acc'][-1]
    poly_results[degree] = val_acc
    print(f"degree={degree}, val_acc={val_acc:.4f}")
    if val_acc > best_poly_acc:
        best_poly_acc = val_acc
        best_poly_model = model
        best_poly_history = history
        best_poly_config = {'degree': degree}

# Save best poly model state
if best_poly_model is not None:
    best_poly_model.save_state_dict(os.path.join(SAVED_MODELS_DIR, "softmax_poly.npz"))
    print("Saved best polynomial softmax model.")


### Model 4 — Softmax Regression + Gaussian Basis Features
We transform inputs with Gaussian RBF features (random centers) and feed them to a softmax classifier.
Tune `n_centers` and `sigma`.


In [ ]:
gauss_results = {}
best_gauss_model = None
best_gauss_history = None
best_gauss_acc = -np.inf
best_gauss_config = None

for centers in GAUSSIAN_CENTERS:
    for sigma in GAUSSIAN_SIGMAS:
        print(f"\n--- centers={centers}, sigma={sigma} ---")
        rbf = GaussianBasisFeatures(n_centers=centers, sigma=sigma, random_state=SEED)
        X_train_rbf = rbf.fit_transform(X_train)
        X_val_rbf = rbf.transform(X_val)
        print("Transformed shapes:", X_train_rbf.shape, X_val_rbf.shape)

        model = build_softmax_model(X_train_rbf.shape[1])
        loss_fn = CrossEntropyLoss(reduction='mean')
        history = train_autograd(model, X_train_rbf, y_train, X_val_rbf, y_val,
                                 loss_fn, lr=LR_AUTOGRAD, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True)
        val_acc = history['val_acc'][-1]
        gauss_results[(centers, sigma)] = val_acc
        print(f"centers={centers}, sigma={sigma} val_acc={val_acc:.4f}")
        if val_acc > best_gauss_acc:
            best_gauss_acc = val_acc
            best_gauss_model = model
            best_gauss_history = history
            best_gauss_config = {'centers': centers, 'sigma': sigma}

# Save best gaussian model state
if best_gauss_model is not None:
    best_gauss_model.save_state_dict(os.path.join(SAVED_MODELS_DIR, "softmax_rbf.npz"))
    print("Saved best gaussian softmax model.")


### Model 5 — MLP (Sequential with Linear + ReLU)
We'll build a simple MLP and tune hidden sizes and learning rate.

In [ ]:
mlp_results = {}
best_mlp_model = None
best_mlp_history = None
best_mlp_acc = -np.inf
best_mlp_config = None

# Example configs to try
MLP_HIDDEN_SIZES = [128, 256, 512]
MLP_LRS = [0.05, 0.01]

print("Training MLPs with different configurations...")

for hidden_size in MLP_HIDDEN_SIZES:
    for lr in MLP_LRS:
        print(f"\n--- hidden_size={hidden_size}, lr={lr} ---")

        input_dim = X_train.shape[1]
        num_classes = len(np.unique(y_train))

        mlp = Sequential(
            Linear(input_dim, hidden_size),
            ReLU(),
            Linear(hidden_size, num_classes),
            Softmax()
        )

        criterion = CrossEntropyLoss()
        optimizer = SGD(list(mlp.parameters()), lr=lr)

        epochs = 30
        batch_size = 64
        train_losses, val_accuracies = [], []

        for epoch in range(epochs):
            perm = np.random.permutation(len(X_train))
            Xs, ys = X_train[perm], y_train[perm]
            epoch_loss = 0.0

            for i in range(0, len(Xs), batch_size):
                xb = Xs[i:i+batch_size]
                yb = ys[i:i+batch_size]

                logits = mlp(Value(xb))
                loss = criterion(logits, yb)
                epoch_loss += loss.data

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Validation
            val_logits = mlp(Value(X_val))
            val_preds = np.argmax(val_logits.data, axis=1)
            val_acc = np.mean(val_preds == y_val)

            train_losses.append(epoch_loss / (len(Xs) / batch_size))
            val_accuracies.append(val_acc)

            print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {train_losses[-1]:.4f} | Val Acc: {val_acc:.4f}")

        # Store results
        mlp_results[(hidden_size, lr)] = val_accuracies[-1]

        # Save best
        if val_accuracies[-1] > best_mlp_acc:
            best_mlp_acc = val_accuracies[-1]
            best_mlp_model = mlp
            best_mlp_history = {'train_loss': train_losses, 'val_acc': val_accuracies}
            best_mlp_config = {'hidden_size': hidden_size, 'lr': lr}

        print(f"hidden_size={hidden_size}, lr={lr} val_acc={val_accuracies[-1]:.4f}")

# Save best MLP model
if best_mlp_model is not None:
    best_mlp_model.save_state_dict(os.path.join(SAVED_MODELS_DIR, "mlp.npz"))
    print("Saved best mlp model.")

### Summary of best validation results per model
We'll collect the validation accuracies and best hyperparameters discovered above.


In [ ]:
results_summary = []

# OvR
results_summary.append({
    'model': 'OvR Logistic',
    'best_valid_acc': float(ovr_val_acc),
    'best_params': {'alpha': best_alpha_ovr},
    'saved_path': ovr_pickle_path
})

# Softmax raw
results_summary.append({
    'model': 'Softmax (raw)',
    'best_valid_acc': float(best_softmax_acc),
    'best_params': best_softmax_config,
    'saved_path': os.path.join(SAVED_MODELS_DIR, "softmax_raw.npz")
})

# Softmax poly
results_summary.append({
    'model': 'Softmax + Poly',
    'best_valid_acc': float(best_poly_acc),
    'best_params': best_poly_config,
    'saved_path': os.path.join(SAVED_MODELS_DIR, "softmax_poly.npz")
})

# Softmax rbf
results_summary.append({
    'model': 'Softmax + RBF',
    'best_valid_acc': float(best_gauss_acc),
    'best_params': best_gauss_config,
    'saved_path': os.path.join(SAVED_MODELS_DIR, "softmax_rbf.npz")
})

# MLP
results_summary.append({
    'model': 'MLP',
    'best_valid_acc': float(best_mlp_acc),
    'best_params': best_mlp_config,
    'saved_path': os.path.join(SAVED_MODELS_DIR, "mlp_best.npz")
})

# Display results summary
from pprint import pprint
pprint(results_summary)


### Final evaluation of each chosen model on the held-out test set
We will evaluate each best model on the original 10k test that we loaded earlier.

In [ ]:
final_results = []

# OvR on test
ovr_test_preds = ovr_predict(ovr_classifiers, X_test_full)
ovr_test_acc = np.mean(ovr_test_preds == y_test_full)
final_results.append({'model': 'OvR Logistic', 'test_acc': float(ovr_test_acc)})

# Softmax raw test
softmax_test_acc = eval_autograd(best_softmax, X_test_full, y_test_full, batch_size=BATCH_SIZE)
final_results.append({'model': 'Softmax (raw)', 'test_acc': softmax_test_acc})

# Softmax poly
if best_poly_model is not None:
    deg = best_poly_config['degree']
    poly = PolynomialFeatures(degree=deg, include_bias=False)
    poly.fit(X_train)
    X_test_poly = poly.transform(X_test_full)
    poly_test_acc = eval_autograd(best_poly_model, X_test_poly, y_test_full, batch_size=BATCH_SIZE)
else:
    poly_test_acc = None
final_results.append({'model': 'Softmax + Poly', 'test_acc': float(poly_test_acc) if poly_test_acc is not None else None})

# Softmax rbf test
if best_gauss_model is not None:
    centers = best_gauss_config['centers']
    sigma = best_gauss_config['sigma']
    rbf = GaussianBasisFeatures(n_centers=centers, sigma=sigma, random_state=SEED)
    rbf.fit(X_train)
    X_test_rbf = rbf.transform(X_test_full)
    rbf_test_acc = eval_autograd(best_gauss_model, X_test_rbf, y_test_full, batch_size=BATCH_SIZE)
else:
    rbf_test_acc = None
final_results.append({'model': 'Softmax + RBF', 'test_acc': float(rbf_test_acc) if rbf_test_acc is not None else None})

# MLP test
if best_mlp_model is not None:
    mlp_test_acc = eval_autograd(best_mlp_model, X_test_full, y_test_full, batch_size=BATCH_SIZE)
else:
    mlp_test_acc = None
final_results.append({'model': 'MLP', 'test_acc': float(mlp_test_acc) if mlp_test_acc is not None else None})

pprint(final_results)


### Results Table & Plot
- A simple table showing test accuracy for the 5 models.
- Plot training/validation curves for autograd models (Softmax raw, Poly, RBF, MLP).


In [ ]:
# ============================================================
#    Capstone Showdown Analysis — Training Loss Curves
# ============================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

if best_softmax_history:
    plt.plot(best_softmax_history['train_loss'], label='Softmax (Raw)')
if best_poly_history:
    plt.plot(best_poly_history['train_loss'], label='Softmax (Poly)')
if best_gauss_history:
    plt.plot(best_gauss_history['train_loss'], label='Softmax (RBF)')
if best_mlp_history:
    plt.plot(best_mlp_history['train_loss'], label='MLP')

plt.title("Training Loss Curves — Capstone Showdown")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

results_table = pd.DataFrame({
    "Model": ["Softmax (Raw)", "Softmax (Poly)", "Softmax (RBF)", "MLP"],
    "Best Val Accuracy (%)": [
        round(best_softmax_history['val_acc'][-1]*100, 2) if best_softmax_history else None,
        round(best_poly_history['val_acc'][-1]*100, 2) if best_poly_history else None,
        round(best_gauss_history['val_acc'][-1]*100, 2) if best_gauss_history else None,
        round(best_mlp_history['val_acc'][-1]*100, 2) if best_mlp_history else None
    ]
})

display(results_table)


In [ ]:
import pandas as pd
df = pd.DataFrame(final_results)
display(df)

# Plot validation curves for autograd models (if histories exist)
plt.figure(figsize=(10,6))
if best_softmax_history:
    plt.plot(best_softmax_history['val_acc'], label='Softmax raw')
if best_poly_history:
    plt.plot(best_poly_history['val_acc'], label='Softmax + Poly')
if best_gauss_history:
    plt.plot(best_gauss_history['val_acc'], label='Softmax + RBF')
if best_mlp_history:
    plt.plot(best_mlp_history['val_acc'], label='MLP')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Validation Accuracy Curves')
plt.legend()
plt.grid(True)
plt.show()
